In [ ]:
import pandas as pd
import numpy as np
import pyreadr
import plotly.express as px
import plotly.graph_objects as go
import os
import time
import requests
from scipy.interpolate import UnivariateSpline
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Creating bedroom timeseries

## 2021 census data 

### number of bedrooms 2021

In [ ]:
bedrooms_2021 = pd.read_csv('/Users/user1/Downloads/bedrooms_2021.csv', skiprows=7)
#remove last 5 rows
bedrooms_2021 = bedrooms_2021[:-5]

In [ ]:
bedrooms_2021 

## 2011 data census

### number of bedrooms 2021

In [ ]:
bedrooms_2011 = pd.read_csv('/Users/user1/Downloads/bedrooms_2011.csv', skiprows=7)
#remove last 5 rows
bedrooms_2011 = bedrooms_2011[:-4]

In [ ]:
bedrooms_2011

### Totall number of bedrooms change at oa

In [ ]:
bedrooms_2011

In [ ]:
bedrooms_2021

## Total bedrooms calculate

In [ ]:
def calculate_total_bedrooms_2021(df, multiplier=4):
    """
    Simple function to add total bedrooms column to 2021 census data

    Parameters:
    df (DataFrame): DataFrame with bedroom category columns
    multiplier (int): What to multiply "5 or more bedrooms" by
    
    Returns:
    DataFrame: DataFrame with total_bedrooms_2021 column added
    """
    
    df = df.copy()
    
    # Calculate total bedrooms
    df['total_bedrooms_2021'] = (
        df.get('No bedrooms', 0) * 0 +
        df.get('1 bedroom', 0) * 1 +
        df.get('2 bedrooms', 0) * 2 +
        df.get('3 bedrooms', 0) * 3 +
        df.get('4 or more bedrooms', 0) * multiplier
    )

    print(f"Total bedrooms calculated using {multiplier} for '4+ bedrooms'")

    return df



In [ ]:
def calculate_total_bedrooms_2011(df, multiplier=5):
    """
    Simple function to add total bedrooms column to 2011 census data

    Parameters:
    df (DataFrame): DataFrame with bedroom category columns
    multiplier (int): What to multiply "4 or more bedrooms" by

    Returns:
    DataFrame: DataFrame with total_bedrooms_2011 column added
    """
    
    df = df.copy()
    
    # Calculate total bedrooms
    df['total_bedrooms_2011'] = (
        df.get('No bedrooms', 0) * 0 +
        df.get('1 bedroom', 0) * 1 +
        df.get('2 bedrooms', 0) * 2 +
        df.get('3 bedrooms', 0) * 3 +
        df.get('4 bedrooms', 0) * 4 +
        df.get('5 or more bedrooms', 0) * multiplier
    )
    
    print(f"Total bedrooms calculated using {multiplier} for '5+ bedrooms'")
    
    return df


In [ ]:
bedrooms_2011_totals = calculate_total_bedrooms_2011(bedrooms_2011)

In [ ]:
bedrooms_2021_totals = calculate_total_bedrooms_2021(bedrooms_2021)

In [ ]:
bedrooms_2011_totals

In [ ]:
bedrooms_2021_totals

In [ ]:
# Combine '4 bedrooms' and '5 or more bedrooms' into a new '4 or more bedrooms' column in bedrooms_2011
bedrooms_2011_totals['4 or more bedrooms'] = bedrooms_2011_totals['4 bedrooms'] + bedrooms_2011_totals['5 or more bedrooms']
#drop the old columns
bedrooms_2011_totals.drop(columns=['No bedrooms','4 bedrooms', '5 or more bedrooms'], inplace=True)
#drop unnessacry columns All categories: Number of bedrooms
bedrooms_2011_totals.drop(columns=['All categories: Number of bedrooms'], inplace=True)
#drop Total: All households	
bedrooms_2021_totals.drop(columns=['Total: All households'], inplace=True)


In [ ]:
#rename 2011 output area code to OA11CD
bedrooms_2011_totals.rename(columns={'2011 output area': 'OA11CD'}, inplace=True)
bedrooms_2021_totals.rename(columns={'2021 output area': 'OA21CD'}, inplace=True)

## converting 2011 output area totals to ward 2022 with weighting for total bedroom values

In [ ]:
oa_2011_lookup_weighting = pd.read_csv('/Users/user1/Downloads/oa_ward_address_lookup_2011 1(in).csv')
oa_2021_lookup_weighting = pd.read_csv('/Users/user1/Downloads/oa_ward_address_lookup_2021 2 1(in).csv')

In [ ]:
oa_2011_lookup_weighting 

In [ ]:
oa_2021_lookup_weighting 

In [ ]:
#rename gss_code_oa_11 to OA11CD
oa_2011_lookup_weighting.rename(columns={'oa_11': 'OA11CD'}, inplace=True)

#rename gss_code_oa_21 to OA21CD
oa_2021_lookup_weighting.rename(columns={'gss_code_oa_21': 'OA21CD'}, inplace=True)


In [ ]:
#bedrooms_2011_totals = bedrooms_2011_totals[['OA11CD','total_bedrooms_2011']]

In [ ]:
bedrooms_2011_totals

In [ ]:
bedrooms_2021_totals

In [ ]:
oa_2011_lookup_weighting

In [ ]:
# join oa_2011_lookup_weighting with bedrooms_2011_totals on OA11CD
bedrooms_2011_totals_merge = bedrooms_2011_totals.merge(oa_2011_lookup_weighting, on='OA11CD', how='left')

In [ ]:
bedrooms_2011_totals_merge

#### create 2011 bedroom weighted totals at ward 2022 level

In [ ]:
# List of bedroom category columns to weight
bedroom_columns = ['1 bedroom', '2 bedrooms', '3 bedrooms', '4 or more bedrooms', 'total_bedrooms_2011']

# Apply the weight to each bedroom category column
for col in bedroom_columns:
    weighted_col = f'weighted_{col.replace(" ", "_").replace("-", "_").lower()}'
    bedrooms_2011_totals_merge[weighted_col] = bedrooms_2011_totals_merge[col] * bedrooms_2011_totals_merge['oa_11_ward_22_weight']

# Group by ward to sum each weighted bedroom column
ward_bedroom_totals = bedrooms_2011_totals_merge.groupby('gss_code_ward_22')[
    [f'weighted_{col.replace(" ", "_").replace("-", "_").lower()}' for col in bedroom_columns]
].sum().reset_index()

# Optionally rename the weighted columns to indicate totals at the ward level
ward_bedroom_totals.rename(columns={
    f'weighted_{col.replace(" ", "_").replace("-", "_").lower()}': f'total_ward_{col.replace(" ", "_").replace("-", "_").lower()}_2011'
    for col in bedroom_columns
}, inplace=True)

# Merge the totals back to the main DataFrame
bedrooms_2011_totals_merge = bedrooms_2011_totals_merge.merge(
    ward_bedroom_totals, 
    on='gss_code_ward_22', 
    how='left'
)


In [ ]:
# # Create the weighted bedroom column weighted by the overlay of oa 2011 into ward 2021
# bedrooms_2011_totals_merge['weighted_total_bedroom_counts_oa11'] = (
#     bedrooms_2011_totals_merge['total_bedrooms_2011'] * bedrooms_2011_totals_merge['oa_11_ward_22_weight']
# )

# # Group by ward code to sum the weighted totals (create separate DataFrame)
# ward_bedroom_totals = bedrooms_2011_totals_merge.groupby('gss_code_ward_22')['weighted_total_bedroom_counts_oa11'].sum().reset_index()
# ward_bedroom_totals.rename(columns={'weighted_total_bedroom_counts_oa11': 'total_ward_bedrooms_2011'}, inplace=True)

# # Merge the ward totals back into the original DataFrame
# bedrooms_2011_totals_merge = bedrooms_2011_totals_merge.merge(
#     ward_bedroom_totals, 
#     on='gss_code_ward_22', 
#     how='left'
# )

In [ ]:
bedrooms_2011_totals_merge

In [ ]:
bedrooms_2011_totals_merge.columns.to_list()

## converting 2021 output area totals to ward 2022 with weighting for total bedroom values

In [ ]:
#bedrooms_2021_totals = bedrooms_2021_totals[['OA21CD','total_bedrooms_2021']]

In [ ]:
bedrooms_2021_totals

In [ ]:
oa_2021_lookup_weighting

#### create 2021 bedroom weighted totals at ward 2022 level

In [ ]:
# Join OA 2021 lookup with bedrooms 2021 totals on OA21CD
bedrooms_2021_totals_merge = bedrooms_2021_totals.merge(oa_2021_lookup_weighting, on='OA21CD', how='left')

# List of bedroom category columns to apply weighting to
bedroom_columns_2021 = ['1 bedroom', '2 bedrooms', '3 bedrooms', '4 or more bedrooms', 'total_bedrooms_2021']

# Apply the weight to each bedroom category column
for col in bedroom_columns_2021:
    weighted_col = f'weighted_{col.replace(" ", "_").replace("-", "_").lower()}'
    bedrooms_2021_totals_merge[weighted_col] = bedrooms_2021_totals_merge[col] * bedrooms_2021_totals_merge['oa_21_ward_22_weight']

# Group by ward to sum each weighted bedroom column
ward_bedroom_totals_2021 = bedrooms_2021_totals_merge.groupby('gss_code_ward_22')[
    [f'weighted_{col.replace(" ", "_").replace("-", "_").lower()}' for col in bedroom_columns_2021]
].sum().reset_index()

# Rename columns to indicate they are ward-level totals for 2021
ward_bedroom_totals_2021.rename(columns={
    f'weighted_{col.replace(" ", "_").replace("-", "_").lower()}': f'total_ward_{col.replace(" ", "_").replace("-", "_").lower()}_2021'
    for col in bedroom_columns_2021
}, inplace=True)

# Merge the ward totals back into the original DataFrame
bedrooms_2021_totals_merge = bedrooms_2021_totals_merge.merge(
    ward_bedroom_totals_2021, 
    on='gss_code_ward_22', 
    how='left'
)


In [ ]:
#  join oa_2021_lookup_weighting with bedrooms_2021_totals on OA21CD
# bedrooms_2021_totals_merge = bedrooms_2021_totals.merge(oa_2021_lookup_weighting, on='OA21CD', how='left')

# # Create the weighted bedroom column weighted by the overlay of oa 2021 into ward 2021
# bedrooms_2021_totals_merge['weighted_total_bedroom_counts_oa21'] = (
#     bedrooms_2021_totals_merge['total_bedrooms_2021'] * bedrooms_2021_totals_merge['oa_21_ward_22_weight']
# )

# # Group by ward code to sum the weighted totals (create separate DataFrame)
# ward_bedroom_totals = bedrooms_2021_totals_merge.groupby('gss_code_ward_22')['weighted_total_bedroom_counts_oa21'].sum().reset_index()
# ward_bedroom_totals.rename(columns={'weighted_total_bedroom_counts_oa21': 'total_ward_bedrooms_2021'}, inplace=True)

# # Merge the ward totals back into the original DataFrame
# bedrooms_2021_totals_merge = bedrooms_2021_totals_merge.merge(
#     ward_bedroom_totals, 
#     on='gss_code_ward_22', 
#     how='left'
# )

In [ ]:
bedrooms_2011_totals_merge.columns.to_list()

In [ ]:
#filter till 'weighted_1_bedroom',
bedrooms_2011_totals_merge_filter = bedrooms_2011_totals_merge[[
'gss_code_ward_22',
'total_ward_1_bedroom_2011',
'total_ward_2_bedrooms_2011',
'total_ward_3_bedrooms_2011',
'total_ward_4_or_more_bedrooms_2011',
'total_ward_total_bedrooms_2011_2011']]



In [ ]:
bedrooms_2011_totals_merge_filter

In [ ]:
bedrooms_2011_totals_merge_filter

In [ ]:
bedrooms_2021_totals_merge_filter = bedrooms_2021_totals_merge[[
    'gss_code_ward_22',
    'total_ward_1_bedroom_2021',
    'total_ward_2_bedrooms_2021',
    'total_ward_3_bedrooms_2021',
    'total_ward_4_or_more_bedrooms_2021',
    'total_ward_total_bedrooms_2021_2021']]

In [ ]:
bedrooms_2021_totals_merge_filter

## Join bedroom and address data for ward

In [ ]:
## add year column bedroom data
bedrooms_2011_totals_merge_filter['year'] = 2011
bedrooms_2021_totals_merge_filter['year'] = 2021

In [ ]:
#read in address counts
oa_and_ward_address_counts = pd.read_csv('/Users/user1/Documents/recoding_adress_data_2011_2021/london_data/oa_adress_counts.csv')

In [ ]:
# Filter to only include the columns we need
oa_and_ward_address_counts_ward =oa_and_ward_address_counts[['ward22cd','year', 'weighted_address_count_ward22']]

In [ ]:
oa_and_ward_address_counts_ward

In [ ]:
#rename gss_code_ward_22 to wd22cd
oa_and_ward_address_counts_ward = oa_and_ward_address_counts_ward.rename(columns={'ward22cd': 'wd22cd'})

In [ ]:
#Standardise and combine 2011 and 2021 bedroom type data
def standardise_bedroom_columns(df, year):
    rename_dict = {
        f'total_ward_1_bedroom_{year}': 'total_ward_1_bedroom',
        f'total_ward_2_bedrooms_{year}': 'total_ward_2_bedrooms',
        f'total_ward_3_bedrooms_{year}': 'total_ward_3_bedrooms',
        f'total_ward_4_or_more_bedrooms_{year}': 'total_ward_4_or_more_bedrooms',
        f'total_ward_total_bedrooms_{year}_{year}': 'total_bedrooms',
        'gss_code_ward_22': 'wd22cd'
    }
    df = df.rename(columns=rename_dict)
    df['year'] = int(year)
    return df

# Apply standardization
bedrooms_2011_clean = standardise_bedroom_columns(bedrooms_2011_totals_merge_filter, 2011)
bedrooms_2021_clean = standardise_bedroom_columns(bedrooms_2021_totals_merge_filter, 2021)


In [ ]:
# Combine into long format
bedroom_data_combined = pd.concat([bedrooms_2011_clean, bedrooms_2021_clean], ignore_index=True)

In [ ]:
bedroom_data_combined

In [ ]:
oa_and_ward_address_counts_ward

In [ ]:
merged_address_and_bedroom_data = oa_and_ward_address_counts_ward.merge(bedroom_data_combined, on=['wd22cd', 'year'], how='left')

In [ ]:
merged_address_and_bedroom_data

In [ ]:
# what years are available in the address data
print("Years available in address data:")
print(sorted(oa_and_ward_address_counts_ward['year'].unique()))

print("\nYears available in bedroom data:")
print(sorted(bedroom_data_combined['year'].unique()))

# Create a complete dataset with all years from address data
# Keep all address data and do a LEFT join with bedroom data
final_merged_data_complete = oa_and_ward_address_counts_ward.merge(
    bedroom_data_combined, 
    on=['year', 'wd22cd'], 
    how='left'  # This keeps all address data years
)

# Fill missing bedroom values for years without bedroom data
# You can either fill with NaN or interpolate/forward-fill
print(f"\n✅ Complete merged data shape: {final_merged_data_complete.shape}")
print(f"📊 All years included: {sorted(final_merged_data_complete['year'].unique())}")
print(f"📊 Unique wards: {final_merged_data_complete['wd22cd'].nunique()}")

# Check missing bedroom data by year
print(f"\n🔍 Records per year:")
print(final_merged_data_complete['year'].value_counts().sort_index())

print(f"\n🔍 Missing bedroom data by year:")
missing_bedrooms = merged_address_and_bedroom_data['total_bedrooms'].isna().groupby(merged_address_and_bedroom_data['year']).sum()
print(missing_bedrooms)

In [ ]:
merged_address_and_bedroom_data

In [ ]:
final_merged_data_complete_drop_duplicates = merged_address_and_bedroom_data.drop_duplicates()

In [ ]:
final_merged_data_complete_drop_duplicates

In [ ]:
final_merged_data_complete_drop_duplicates['wd22cd'].nunique()

In [ ]:
final_merged_data_complete_drop_duplicates

### interpolating missing bedroom yeah
### generate a spline
### us raito from known values

In [ ]:
# Bedroom type columns to interpolate
bedroom_types = [
    'total_ward_1_bedroom',
    'total_ward_2_bedrooms',
    'total_ward_3_bedrooms',
    'total_ward_4_or_more_bedrooms'
]

In [ ]:
def interpolate_bedroom_types(group, bedroom_types, spline_smooth=0):
    group = group.sort_values('year')

    # Fit spline to address count
    addr_clean = group.dropna(subset=['year', 'weighted_address_count_ward22'])
    if len(addr_clean) < 2:
        return group

    x = addr_clean['year'].values
    y = addr_clean['weighted_address_count_ward22'].values
    spline = UnivariateSpline(x, y, s=spline_smooth)
    interpolated_addresses = spline(group['year'].values)

    # Interpolate each bedroom type
    for bed in bedroom_types:
        known = group.dropna(subset=[bed, 'weighted_address_count_ward22'])
        if len(known) < 2:
            continue

        ratio_years = known['year'].values
        ratios = (known[bed] / known['weighted_address_count_ward22']).values
        interpolated_ratios = np.interp(group['year'].values, ratio_years, ratios)

        estimated = interpolated_ratios * interpolated_addresses
        group[bed] = group[bed].combine_first(pd.Series(estimated, index=group.index))

    return group


In [ ]:
from scipy.interpolate import UnivariateSpline
import numpy as np
import pandas as pd

def interpolate_bedrooms_with_spline(group, spline_smooth=0):
    # Sort for safety
    group = group.sort_values('year')

    # STEP 1: Clean address data
    addr_clean = group.dropna(subset=['year', 'weighted_address_count_ward22'])
    if len(addr_clean) < 2:
        return group  # Not enough data for spline

    x = addr_clean['year'].values
    y = addr_clean['weighted_address_count_ward22'].values

    # Fit spline to address counts
    spline = UnivariateSpline(x, y, s=spline_smooth)
    interpolated_addresses = spline(group['year'].values)

    # STEP 2: Get bedroom/address ratios
    known = group.dropna(subset=['total_ward_bedrooms', 'weighted_address_count_ward22'])
    if len(known) < 2:
        return group  # Not enough to interpolate ratio

    ratio_years = known['year'].values
    ratios = (known['total_bedrooms'] / known['weighted_address_count_ward22']).values

    # Interpolate bedroom-per-address ratio linearly
    interpolated_ratios = np.interp(group['year'].values, ratio_years, ratios)

    # STEP 3: Estimate bedrooms
    estimated_bedrooms = interpolated_ratios * interpolated_addresses

    group['total_bedrooms'] = group['total_bedrooms'].combine_first(
        pd.Series(estimated_bedrooms, index=group.index)
    )

    return group




In [ ]:
final_merged_data_complete_drop_duplicates_interpolated = (
    final_merged_data_complete_drop_duplicates
    .groupby('wd22cd', group_keys=False)
    .apply(interpolate_bedroom_types, bedroom_types=bedroom_types)
)

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated 

In [ ]:
#fill in totals 
# Define bedroom type columns
bedroom_type_columns = [
    'total_ward_1_bedroom',
    'total_ward_2_bedrooms',
    'total_ward_3_bedrooms',
    'total_ward_4_or_more_bedrooms'
]


# Add total_bedrooms column based on bedroom type columns
final_merged_data_complete_drop_duplicates_interpolated['total_bedrooms'] = (
    final_merged_data_complete_drop_duplicates_interpolated['total_ward_1_bedroom'] * 1 +
    final_merged_data_complete_drop_duplicates_interpolated['total_ward_2_bedrooms'] * 2 +
    final_merged_data_complete_drop_duplicates_interpolated['total_ward_3_bedrooms'] * 3 +
    final_merged_data_complete_drop_duplicates_interpolated['total_ward_4_or_more_bedrooms'] * 4
)


In [ ]:
final_merged_data_complete_drop_duplicates_interpolated

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated 

In [ ]:
# Calculate year-on-year (yoy) change for weighted_address_count and weighted_bedroom_count
final_merged_data_complete_drop_duplicates = final_merged_data_complete_drop_duplicates_interpolated.sort_values(['wd22cd', 'year'])

final_merged_data_complete_drop_duplicates['weighted_address_count_yoy'] = (
    final_merged_data_complete_drop_duplicates_interpolated.groupby('wd22cd')['weighted_address_count_ward22'].diff()
)

final_merged_data_complete_drop_duplicates['weighted_bedroom_count_yoy'] = (
    final_merged_data_complete_drop_duplicates_interpolated.groupby('wd22cd')['weighted_address_count_ward22'].diff()
)

In [ ]:
final_merged_data_complete_drop_duplicates
#save the final DataFrame to a CSV file
final_merged_data_complete_drop_duplicates.to_csv('/Users/user1/Documents/recoding_adress_data_2011_2021/london_data/bedroom_count_ward22.csv')

In [ ]:
final_merged_data_complete_drop_duplicates

## Plotting

In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = final_merged_data_complete_drop_duplicates_interpolated.copy()

# Bedroom-related columns
bedroom_cols = [
    'total_ward_1_bedroom',
    'total_ward_2_bedrooms',
    'total_ward_3_bedrooms',
    'total_ward_4_or_more_bedrooms',
    'total_bedrooms'
]

# Get unique ward codes
wards = df['wd22cd'].unique()

# Initialise the figure
fig = go.Figure()

# Add traces for each ward, one trace per bedroom column
for ward in wards:
    ward_data = df[df['wd22cd'] == ward].sort_values('year')
    for col in bedroom_cols:
        fig.add_trace(
            go.Scatter(
                x=ward_data['year'],
                y=ward_data[col],
                mode='lines+markers',
                name=f"{ward} - {col}",
                visible=False
            )
        )

# Make first ward visible by default
for i, trace in enumerate(fig.data):
    if i < len(bedroom_cols):  # first ward
        trace.visible = True

# Create dropdown buttons for each ward
dropdown_buttons = []
for i, ward in enumerate(wards):
    visible = [False] * len(fig.data)
    for j in range(len(bedroom_cols)):
        visible[i * len(bedroom_cols) + j] = True
    dropdown_buttons.append(
        dict(label=ward,
             method='update',
             args=[{'visible': visible},
                   {'title': f"Bedroom Types Over Time for Ward {ward}"}])
    )

# Update layout
fig.update_layout(
    title=f"Bedroom Types Over Time for Ward {wards[0]}",
    xaxis_title='Year',
    yaxis_title='Number of Bedrooms',
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction='down',
            showactive=True,
            x=1.05,
            xanchor='left',
            y=1,
            yanchor='top'
        )
    ],
    legend_title='Bedroom Type'
)

fig.show()


### Specific ward checks

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated
#filter ward E05009291
final_merged_data_complete_drop_duplicates_interpolated_E05009291 = final_merged_data_complete_drop_duplicates_interpolated[final_merged_data_complete_drop_duplicates_interpolated['wd22cd'] == 'E05009291']

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated_E05009291_2021 = final_merged_data_complete_drop_duplicates_interpolated_E05009291[final_merged_data_complete_drop_duplicates_interpolated_E05009291['year'] == 2021]

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated_E05009291_2021

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated_E05009291_2022 = final_merged_data_complete_drop_duplicates_interpolated_E05009291[final_merged_data_complete_drop_duplicates_interpolated_E05009291['year'] == 2022]

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated_E05009291_2022 

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Assuming your dataframe is named df
# Replace this with loading your actual data
# df = pd.read_csv('your_file.csv')

# Convert year to string for better dropdown handling if needed
df = final_merged_data_complete_drop_duplicates.copy()
df['year'] = df['year'].astype(str)

# Get unique ward codes
wards = df['wd22cd'].unique()

# Create traces for each ward but make only one visible by default
fig = go.Figure()

for i, ward in enumerate(wards):
    ward_df = df[df['wd22cd'] == ward]
    fig.add_trace(go.Scatter(
        x=ward_df['year'],
        y=ward_df['weighted_address_count_ward22'],
        mode='lines+markers',
        name=f'{ward} - Address Count',
        visible=(i == 0)
    ))
    fig.add_trace(go.Scatter(
        x=ward_df['year'],
        y=ward_df['total_bedrooms'],
        mode='lines+markers',
        name=f'{ward} - Bedrooms',
        visible=(i == 0)
    ))

# Create dropdown buttons
dropdown_buttons = []
for i, ward in enumerate(wards):
    visible = [False] * len(wards) * 2
    visible[2*i] = True
    visible[2*i + 1] = True
    dropdown_buttons.append(dict(
        label=ward,
        method='update',
        args=[{'visible': visible},
              {'title': f'Data for Ward {ward}'}]
    ))

# Update layout
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=dropdown_buttons,
        x=0.1,
        y=1.2,
        xanchor='left',
        yanchor='top'
    )],
    title='Ward Data Over Time',
    xaxis_title='Year',
    yaxis_title='Count',
    legend_title='Metric'
)

fig.show()


# sum the bedroom totals but weighted


In [ ]:
final_merged_data_complete_drop_duplicates_interpolated

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated

In [ ]:
# Round to nearest integer
final_merged_data_complete_drop_duplicates_interpolated = final_merged_data_complete_drop_duplicates_interpolated.round() 

In [ ]:
# change in bedroom from 2011 to 2021 oa ward level
# change in address yoy at ward level and oa level
# distribution the change of bedrooms in 2011 and 2021 depending the change in address yoy at ward level and oa level for yoy

## mapping data

In [ ]:
# Read in geojson file for wards
ward_geojson_path = "/Users/user1/Documents/recoding_adress_data_2011_2021/london_data/Wards_December_2022_Boundaries_UK_BSC_-6356914696862559311.geojson"
import geopandas as gpd
ward_gdf = gpd.read_file(ward_geojson_path)


In [ ]:
ward_gdf = ward_gdf[["WD22CD", "WD22NM", "LAD22NM","geometry"]]
#merge to final_merged_data_complete_drop_duplicates_interpolated
ward_gdf = ward_gdf.rename(columns={'WD22CD': 'wd22cd'})
ward_gdf = ward_gdf.rename(columns={'WD22NM': 'wd22nm'})
ward_gdf = ward_gdf.rename(columns={'LAD22NM': 'lad22nm'})

#merge 
mapping_bedroom_data = final_merged_data_complete_drop_duplicates_interpolated.merge(ward_gdf, on='wd22cd', how='left')


In [ ]:
mapping_bedroom_data

In [ ]:
cols = [
    "weighted_address_count_ward22",
    "total_ward_1_bedroom",
    "total_ward_2_bedrooms",
    "total_ward_3_bedrooms",
    "total_ward_4_or_more_bedrooms",
    "total_bedrooms"
]

# loop through each column and create abs and pct change columns
for col in cols:
    mapping_bedroom_data[f"{col}_abs_chg"] = mapping_bedroom_data.groupby("wd22cd")[col].diff()
    mapping_bedroom_data[f"{col}_pct_chg"] = mapping_bedroom_data.groupby("wd22cd")[col].pct_change() * 100

In [ ]:
mapping_bedroom_data.columns.to_list()

In [ ]:
mapping_bedroom_data[(mapping_bedroom_data['wd22nm'] == 'Bedfont') & (mapping_bedroom_data['year'] == 2011)]

In [ ]:
import geopandas as gpd
import plotly.graph_objects as go

# Setup GeoDataFrame
gdf = gpd.GeoDataFrame(mapping_bedroom_data, geometry='geometry')
gdf = gdf.set_crs(epsg=27700, allow_override=True)

# Variables and years
dropdown_vars = [
    'total_ward_1_bedroom',
    'total_ward_2_bedrooms',
    'total_ward_3_bedrooms',
    'total_ward_4_or_more_bedrooms',
    'total_bedrooms',
    'weighted_address_count_ward22_abs_chg',
    'weighted_address_count_ward22_pct_chg',
    'total_ward_1_bedroom_abs_chg',
    'total_ward_1_bedroom_pct_chg',
    'total_ward_2_bedrooms_abs_chg',
    'total_ward_2_bedrooms_pct_chg',
    'total_ward_3_bedrooms_abs_chg',
    'total_ward_3_bedrooms_pct_chg',
    'total_ward_4_or_more_bedrooms_abs_chg',
    'total_ward_4_or_more_bedrooms_pct_chg',
    'total_bedrooms_abs_chg',
    'total_bedrooms_pct_chg'
]
years = sorted(gdf['year'].unique())

# Initial variable/year
init_var = dropdown_vars[0]
init_year = years[0]

# Initial trace
filtered = gdf[gdf['year'] == init_year]
trace = go.Choroplethmapbox(
    geojson=filtered.__geo_interface__,
    locations=filtered.index,
    z=filtered[init_var].fillna(0),
    colorscale="Viridis" if "_chg" not in init_var else "RdBu",
    marker_opacity=0.7,
    marker_line_width=0.5,
    colorbar_title=init_var,
    hovertext=filtered['wd22nm'],
    hoverinfo='text+z'
)

# Build ALL frames (variable × year)
frames = []
for var in dropdown_vars:
    for year in years:
        filtered = gdf[gdf['year'] == year]
        frames.append(go.Frame(
            name=f"{var}-{year}",
            data=[go.Choroplethmapbox(
                geojson=filtered.__geo_interface__,
                locations=filtered.index,
                z=filtered[var].fillna(0),
                colorscale="Viridis" if "_chg" not in var else "RdBu",
                marker_opacity=0.7,
                marker_line_width=0.5,
                colorbar_title=var,
                hovertext=filtered['wd22nm'],
                hoverinfo='text+z'
            )]
        ))

# Slider steps (year selector – just updates the current var)
steps = []
for year in years:
    step = dict(
        method="animate",
        label=str(year),
        args=[
            # This is where the magic happens:
            # when slider is moved, it uses whatever variable is active
            [f"{init_var}-{year}"], 
            {"mode": "immediate",
             "frame": {"duration": 500, "redraw": True},
             "transition": {"duration": 0}}
        ]
    )
    steps.append(step)

slider = [dict(
    active=0,
    currentvalue={"prefix": "Year: "},
    pad={"t": 50},
    steps=steps
)]

# Dropdown menu (variable selector – rewires slider steps)
dropdown_buttons = []
for var in dropdown_vars:
    new_steps = []
    for year in years:
        new_steps.append(dict(
            method="animate",
            label=str(year),
            args=[
                [f"{var}-{year}"],
                {"mode": "immediate",
                 "frame": {"duration": 500, "redraw": True},
                 "transition": {"duration": 0}}
            ]
        ))

    dropdown_buttons.append(dict(
        label=var,
        method="update",
        args=[
            {},  # no trace visibility toggling needed
            {"title": f"Ward Data: {var}",
             "sliders": [dict(active=0,
                              currentvalue={"prefix": "Year: "},
                              pad={"t": 50},
                              steps=new_steps)]}
        ]
    ))

# Final figure
fig = go.Figure(
    data=[trace],
    frames=frames,
    layout=dict(
        title=f"Ward Data: {init_var}",
        mapbox_style="carto-positron",
        mapbox_zoom=9,
        mapbox_center={"lat": gdf.geometry.centroid.y.mean(),
                       "lon": gdf.geometry.centroid.x.mean()},
        margin={"r": 0, "t": 40, "l": 0, "b": 0},
        sliders=slider,
        updatemenus=[
            dict(
                buttons=dropdown_buttons,
                direction="down",
                showactive=True,
                x=0,
                y=1.15,
                xanchor="left",
                yanchor="top"
            ),
            dict(
                type="buttons",
                showactive=False,
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[None,
                              {"frame": {"duration": 500, "redraw": True},
                               "fromcurrent": True,
                               "transition": {"duration": 0}}]
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[[None],
                              {"mode": "immediate",
                               "frame": {"duration": 0},
                               "transition": {"duration": 0}}]
                    )
                ],
                direction="right",
                x=0.3,
                y=1.15,
                xanchor="left",
                yanchor="top"
            )
        ]
    )
)

fig.show()


## Sense check bedroom estimates

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated['wd22cd'].nunique()

In [ ]:
### where total address counts are greater than total bedroom counts
filtered_df = final_merged_data_complete_drop_duplicates_interpolated[
    final_merged_data_complete_drop_duplicates_interpolated['weighted_address_count_ward22'] 
    > final_merged_data_complete_drop_duplicates_interpolated['total_bedrooms']
]

In [ ]:
filtered_df

In [ ]:
### Add wd22nm and lad22nm 

In [ ]:
mapping_unique = mapping_bedroom_data[['wd22cd', 'wd22nm']].drop_duplicates(subset='wd22cd')

In [ ]:
final_merged_data_complete_drop_duplicates_interpolated_names = (
    final_merged_data_complete_drop_duplicates_interpolated
    .merge(
        mapping_unique,  # only keep columns we need
        on='wd22cd',  # merge key
        how='left'    # keep all rows from the main DataFrame
    )
)


In [ ]:
sense_check_df = final_merged_data_complete_drop_duplicates_interpolated_names.copy()

### Bedrooms-per-address check

In [ ]:
sense_check_df['bedrooms_per_address'] = sense_check_df['total_bedrooms'] / sense_check_df['weighted_address_count_ward22']


In [ ]:
sense_check_df.columns.to_list()

### Negative or missing values

In [ ]:
(sense_check_df[['total_ward_1_bedroom', 'total_ward_2_bedrooms', 
     'total_ward_3_bedrooms', 'total_ward_4_or_more_bedrooms','total_bedrooms','weighted_address_count_ward22']] < 0).any()


### Dwelling vs address count

In [ ]:
### Create dwelling total column this is the sum of all bedroom types

sense_check_df['dwelling_count'] = sense_check_df[['total_ward_1_bedroom', 'total_ward_2_bedrooms', 
     'total_ward_3_bedrooms', 'total_ward_4_or_more_bedrooms']].sum(axis=1)


In [ ]:
sense_check_final_df = sense_check_df.copy()


In [ ]:
# Calculate absolute and percentage differences
sense_check_final_df['abs_diff'] = (
    sense_check_final_df['weighted_address_count_ward22'] - sense_check_final_df['dwelling_count']
).abs()

sense_check_final_df['pct_diff'] = (
    sense_check_final_df['abs_diff'] / sense_check_final_df['weighted_address_count_ward22'] * 100
)

# Top 10 by absolute difference
largest_abs = sense_check_final_df.sort_values('abs_diff', ascending=False).head(10)

# Top 10 by percentage difference
largest_pct = sense_check_final_df.sort_values('pct_diff', ascending=False).head(10)

print("Top 10 absolute differences:")
print(largest_abs[['wd22cd', 'wd22nm', 'year',
                   'weighted_address_count_ward22', 'dwelling_count',
                   'abs_diff', 'pct_diff']])

print("\nTop 10 percentage differences:")
print(largest_pct[['wd22cd', 'wd22nm', 'year',
                   'weighted_address_count_ward22', 'dwelling_count',
                   'abs_diff', 'pct_diff']])


In [ ]:
# Calculate absolute and percentage differences
sense_check_final_df['abs_diff'] = (
    sense_check_final_df['weighted_address_count_ward22'] - sense_check_final_df['dwelling_count']
).abs()

sense_check_final_df['pct_diff'] = (
    sense_check_final_df['abs_diff'] / sense_check_final_df['weighted_address_count_ward22'] * 100
)

# Top 10 by absolute difference
largest_abs = sense_check_final_df.sort_values('abs_diff', ascending=False).head(10)

# Top 10 by percentage difference
largest_pct = sense_check_final_df.sort_values('pct_diff', ascending=False).head(10)

print("Top 10 absolute differences:")
print(largest_abs[['wd22cd', 'wd22nm', 'year',
                   'weighted_address_count_ward22', 'dwelling_count',
                   'abs_diff', 'pct_diff']])

print("\nTop 10 percentage differences:")
print(largest_pct[['wd22cd', 'wd22nm', 'year',
                   'weighted_address_count_ward22', 'dwelling_count',
                   'abs_diff', 'pct_diff']])


In [ ]:
# Filter df for pct_diff_dwelling more than 20%


In [ ]:
df_over_20 = sense_check_final_df[sense_check_final_df['pct_diff'] > 20]

print(df_over_20[['wd22cd', 'wd22nm', 'year',
                  'weighted_address_count_ward22', 'dwelling_count',
                  'abs_diff', 'pct_diff']])


In [ ]:
# percenatge changes yoy

In [ ]:
sense_check_final_df['pct_change_dwelling_yoy'] = sense_check_final_df.groupby('wd22cd')['dwelling_count'].pct_change() * 100.
sense_check_final_df['pct_change_address_yoy'] = sense_check_final_df.groupby('wd22cd')['weighted_address_count_ward22'].pct_change() * 100.

In [ ]:
threshold = 20

large_jumps_dwelling = sense_check_final_df[sense_check_final_df['pct_change_dwelling_yoy'].abs() > threshold ]
large_jumps_address = sense_check_final_df[sense_check_final_df['pct_change_address_yoy'].abs() > threshold ]


In [ ]:
print("Large Year-on-Year Dwelling Count Changes:")
print(large_jumps_dwelling[['wd22cd', 'wd22nm', 'year', 'dwelling_count', 'pct_change_dwelling_yoy']])

print("\nLarge Year-on-Year Address Count Changes:")
print(large_jumps_address[['wd22cd', 'wd22nm', 'year', 'weighted_address_count_ward22', 'pct_change_address_yoy']])


In [ ]:
import plotly.express as px

# Filter for years 2011 and 2021
filtered_df = sense_check_final_df[sense_check_final_df['year'].isin([2011, 2021])]

fig = px.histogram(
    filtered_df,
    x='pct_diff',
    nbins=30,
    title='Histogram of % Difference Between Dwelling Count and Address Count (2011 & 2021)',
    labels={'pct_diff': 'Percentage Difference (%)'},
    color_discrete_sequence=['skyblue']
)

fig.update_layout(
    xaxis_title='Percentage Difference (%)',
    yaxis_title='Frequency',
    bargap=0.1
)

fig.show()


In [ ]:
### comparison plots of dwelling counts and address counts plots

In [ ]:
sns.jointplot(
    data=sense_check_final_df,
    x='weighted_address_count_ward22',
    y='dwelling_count',
    kind='reg',  # regression line + scatter
    height=7
)
plt.show()


In [ ]:
df = sense_check_final_df.dropna(subset=['dwelling_count', 'weighted_address_count_ward22'])
mean_counts = (df['dwelling_count'] + df['weighted_address_count_ward22']) / 2
diff_counts = df['dwelling_count'] - df['weighted_address_count_ward22']

plt.figure(figsize=(8,5))
plt.scatter(mean_counts, diff_counts, alpha=0.5)
plt.axhline(np.mean(diff_counts), color='red', linestyle='--')
plt.axhline(np.mean(diff_counts) + 1.96*np.std(diff_counts), color='gray', linestyle='--')
plt.axhline(np.mean(diff_counts) - 1.96*np.std(diff_counts), color='gray', linestyle='--')

plt.xlabel('Mean of Counts')
plt.ylabel('Difference (Dwelling Count - Weighted Address Count)')
plt.title('Bland-Altman Plot')
plt.show()


### Checks Nans

In [ ]:
nan_counts = sense_check_final_df.isna().sum()
print(nan_counts)


In [ ]:
### if weight address count is 0 make all the bedroom columns 0

In [ ]:
bedroom_cols = [
    'total_ward_1_bedroom',
    'total_ward_2_bedrooms',
    'total_ward_3_bedrooms',
    'total_ward_4_or_more_bedrooms',
    'total_bedrooms'
]

mask = sense_check_final_df['weighted_address_count_ward22'] == 0

sense_check_final_df.loc[mask, bedroom_cols] = 0

In [ ]:
import plotly.express as px

df = sense_check_final_df.copy()

# Create highlight column based on existing pct_diff_dwelling
df['highlight'] = df['pct_diff'] > 50

fig = px.scatter(
    df,
    x='weighted_address_count_ward22',
    y='dwelling_count',
    color='highlight',
    color_discrete_map={True: 'red', False: 'blue'},
    hover_data=['wd22nm', 'year', 'pct_diff'],
    labels={
        'weighted_address_count_ward22': 'Weighted Address Count',
        'dwelling_count': 'Dwelling Count',
        'highlight': 'Difference > 50%'
    },
    title='Weighted Address Count vs Dwelling Count with >50% Difference Highlighted'
)

fig.show()


## LBSM api

In [ ]:
def fetch_lbsm_data(limit=5, offset=0, **kwargs):
    """
    Fetch data from LBSM API with flexible parameters
    
    Parameters:
    limit (int): Number of records to fetch
    offset (int): Offset for pagination
    **kwargs: Additional query parameters
    """
    
    base_url = "https://api2.ldn-gis.co.uk/tables/lbsm/uprn"
    params = {'limit': limit, 'offset': offset}
    params.update(kwargs)
    
    try:
        print(f"Fetching data with params: {params}")
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        
        data = response.json()
        
        # Convert to DataFrame
        if isinstance(data, list):
            df = pd.DataFrame(data)
        elif isinstance(data, dict) and 'data' in data:
            df = pd.DataFrame(data['data'])
        else:
            df = pd.DataFrame([data] if isinstance(data, dict) else data)
        
        print(f"✅ Fetched {len(df)} records")
        return df
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Test with different limits
small_sample = fetch_lbsm_data(limit=10)
if small_sample is not None:
    display(small_sample.head())

In [ ]:
def fetch_lbsm_data(limit=5, offset=0, fetch_all=False, batch_size=1000, delay=0.5, **kwargs):
    """
    Fetch data from LBSM API with flexible parameters
    
    Parameters:
    limit (int): Number of records to fetch (ignored if fetch_all=True)
    offset (int): Offset for pagination (ignored if fetch_all=True)
    fetch_all (bool): If True, fetch all available data
    batch_size (int): Batch size when fetch_all=True
    delay (float): Delay between requests when fetch_all=True
    **kwargs: Additional query parameters
    """
    
    base_url = "https://api2.ldn-gis.co.uk/tables/lbsm/uprn"
    
    if not fetch_all:
        # Original single-batch functionality
        params = {'limit': limit, 'offset': offset}
        params.update(kwargs)
        
        try:
            print(f"🔄 Fetching data with params: {params}")
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            
            data = response.json()
            
            # Convert to DataFrame
            if isinstance(data, list):
                df = pd.DataFrame(data)
            elif isinstance(data, dict) and 'data' in data:
                df = pd.DataFrame(data['data'])
            else:
                df = pd.DataFrame([data] if isinstance(data, dict) else data)
            
            print(f"✅ Fetched {len(df)} records")
            return df
            
        except Exception as e:
            print(f"❌ Error: {e}")
            return None
    
    else:
        # Fetch all data functionality
        all_dataframes = []
        current_offset = 0
        
        print(" Fetching ALL data...")
        
        while True:
            params = {'limit': batch_size, 'offset': current_offset}
            params.update(kwargs)
            
            try:
                print(f" Batch: offset={current_offset}, limit={batch_size}")
                response = requests.get(base_url, params=params)
                response.raise_for_status()
                
                data = response.json()
                
                if isinstance(data, list):
                    df = pd.DataFrame(data)
                elif isinstance(data, dict) and 'data' in data:
                    df = pd.DataFrame(data['data'])
                else:
                    df = pd.DataFrame([data] if isinstance(data, dict) else data)
                
                if df.empty or len(df) == 0:
                    break
                
                all_dataframes.append(df)
                print(f"   ✅ {len(df)} records | Total batches: {len(all_dataframes)}")
                
                if len(df) < batch_size:
                    break
                
                current_offset += batch_size
                time.sleep(delay)
                
            except Exception as e:
                print(f"❌ Error: {e}")
                break
        
        if all_dataframes:
            combined_df = pd.concat(all_dataframes, ignore_index=True)
            print(f"Total records: {len(combined_df)}")
            return combined_df
        else:
            return None

In [ ]:
small_sample = fetch_lbsm_data(limit=10)

In [ ]:
small_sample.columns.to_list()  # Display column names for small sample
#display all columns in the small sample
print("🔍 Column names in small sample:", small_sample.columns.tolist())

In [ ]:
# filter small_sample to include only specific columns 'number_habitable_rooms', 'number_habitable_rooms_known', 'total_floor_area_clean', 'total_floor_area_clean_known', 'estimated_floor_count', 'basement_floor',
filtered_sample = small_sample[['uprn','number_habitable_rooms', 'number_habitable_rooms_known', 'total_floor_area_clean', 'total_floor_area_clean_known', 'estimated_floor_count', 'basement_floor']]

In [ ]:
filtered_sample

In [ ]:
url = "https://api2.ldn-gis.co.uk/tables/lbsm/uprn?limit=5"

### Make mapping GIF 

In [ ]:
import imageio
import os

# Directory for frames
os.makedirs("frames", exist_ok=True)

# Loop through years for a single variable (or however you want to animate)
frames = []
for year in years:
    filtered = gdf[gdf['year'] == year]
    fig = go.Figure(go.Choroplethmapbox(
        geojson=filtered.__geo_interface__,
        locations=filtered.index,
        z=filtered[init_var].fillna(0),
        colorscale="Viridis",
        marker_opacity=0.7,
        marker_line_width=0.5
    ))
    fig.update_layout(
        mapbox_style="carto-positron",
        mapbox_zoom=9,
        mapbox_center={"lat": gdf.geometry.centroid.y.mean(),
                       "lon": gdf.geometry.centroid.x.mean()},
        margin={"r":0,"t":0,"l":0,"b":0}
    )

    # Save frame
    filepath = f"frames/{year}.png"
    fig.write_image(filepath, scale=2)  # requires kaleido
    frames.append(imageio.imread(filepath))

# Stitch into gif
imageio.mimsave("ward_animation.gif", frames, duration=1)


In [ ]:
import imageio
import os
import numpy as np

# Directory for frames
os.makedirs("frames", exist_ok=True)

# Determine fixed min/max for the legend
zmin = gdf[init_var].min()
zmax = gdf[init_var].max()

frames = []
for year in years:
    filtered = gdf[gdf['year'] == year]
    fig = go.Figure(go.Choroplethmapbox(
        geojson=filtered.__geo_interface__,
        locations=filtered.index,
        z=filtered[init_var].fillna(0),
        colorscale="Viridis",
        zmin=zmin,  # fixed min
        zmax=zmax,  # fixed max
        marker_opacity=0.7,
        marker_line_width=0.5
    ))
    fig.update_layout(
        mapbox_style="carto-positron",
        mapbox_zoom=9,
        mapbox_center={"lat": gdf.geometry.centroid.y.mean(),
                       "lon": gdf.geometry.centroid.x.mean()},
        margin={"r":0,"t":0,"l":0,"b":0}
    )

    # Save frame
    filepath = f"frames/{year}.png"
    fig.write_image(filepath, scale=2)
    frames.append(imageio.imread(filepath))

# Stitch into gif
imageio.mimsave("ward_animation.gif", frames, duration=3)


In [ ]:
import imageio
import os

# Directory for frames
os.makedirs("frames", exist_ok=True)

# Fixed color scale
zmin = gdf[init_var].min()
zmax = gdf[init_var].max()

frames = []
for year in years:
    filtered = gdf[gdf['year'] == year]
    fig = go.Figure(go.Choroplethmapbox(
        geojson=filtered.__geo_interface__,
        locations=filtered.index,
        z=filtered[init_var].fillna(0),
        colorscale="Viridis",
        zmin=zmin,
        zmax=zmax,
        marker_opacity=0.7,
        marker_line_width=0.5,
        colorbar=dict(title="Total<br>1-Bedroom<br>Dwellings")  # Legend title
    ))

    # Add main figure title
    fig.update_layout(
        title_text=f"Total Number of 1-Bedroom Dwellings in {year}",  # Dynamic year
        mapbox_style="carto-positron",
        mapbox_zoom=9,
        mapbox_center={"lat": gdf.geometry.centroid.y.mean(),
                       "lon": gdf.geometry.centroid.x.mean()},
        margin={"r":0,"t":50,"l":0,"b":0}  # leave space for title
    )

    # Save frame
    filepath = f"frames/{year}.png"
    fig.write_image(filepath, scale=2)
    frames.append(imageio.imread(filepath))


with imageio.get_writer("ward_animation.gif", mode='I', fps=4) as writer:
    for frame in frames:
        writer.append_data(frame)


